# Milestone 2: Data Processing & Transformation
**Project:** Public Health Data Visualization System  
**Dataset:** Global Health Statistics (1,000,000 records × 22 columns)  

**Objective:** Develop a robust, reproducible data processing pipeline that cleans, transforms, and engineers features from the raw dataset.

---
**Assigned to:** *(Ritah, Sharon, Joyce, Christine, Jubilee)*  
**Branch:** `feature/milestone2-pipeline`

> **Before starting:** Run `load_dataset.ipynb` to confirm the dataset is available locally.

---
## 1. Setup & Data Loading

**What to do here:**
- Import all required libraries (`pandas`, `numpy`, `os`, `pathlib`, etc.)
- Load the raw dataset from `data/raw/` using a reproducible file path
- Print the shape and a preview of the data to confirm it loaded correctly

**Deliverable check:** The pipeline must start from a reproducible data source — no hardcoded absolute paths.

In [1]:
# Import libraries
import pandas as pd #for data manipulation
import numpy as np #for numerical operations
from pathlib import Path #for handling file paths(different OS compatibility)

# Define reproducible data path (relative to notebook location)
NOTEBOOK_DIR = Path(__file__).parent.resolve() if '__file__' in dir() else Path.cwd()#notebook directory or cwd if __file__ is unavailable
PROJECT_ROOT = NOTEBOOK_DIR.parent  # Go up from notebooks/ to project root
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Global Health Statistics.csv"#build path to the raw CSV file


# Load dataset
df = pd.read_csv(RAW_PATH)#read the CSV file into a DataFrame

# Validation checks
print("Dataset shape:", df.shape)#print the shape of the dataset (rows, columns)
print("First 5 rows:")#print the first 5 rows of the dataset to verify it loaded correctly
print(df.head())#print the column names to check for any issues with headers

Dataset shape: (1000000, 22)
First 5 rows:
     Country  Year         Disease Name Disease Category  Prevalence Rate (%)  \
0      Italy  2013              Malaria      Respiratory                 0.95   
1     France  2002                Ebola        Parasitic                12.46   
2     Turkey  2015             COVID-19          Genetic                 0.91   
3  Indonesia  2011  Parkinson's Disease       Autoimmune                 4.68   
4      Italy  2013         Tuberculosis          Genetic                 0.83   

   Incidence Rate (%)  Mortality Rate (%) Age Group Gender  \
0                1.55                8.42      0-18   Male   
1                8.63                8.75       61+   Male   
2                2.35                6.22     36-60   Male   
3                6.29                3.99      0-18  Other   
4               13.59                7.01       61+   Male   

   Population Affected  ...  Hospital Beds per 1000  Treatment Type  \
0               471007  ..

---
## 2. Data Quality Audit

**What to do here:**
- Scan for **missing values** — count and percentage per column
- Detect **duplicate rows**
- Identify **statistical anomalies/outliers** in numerical columns (e.g., using IQR or z-score)
- Check for **invalid values** (e.g., percentages outside 0–100, negative values where impossible)
- Check **data types** — are columns stored as the correct type?

**Deliverable check:** This section documents all issues *before* they are fixed. Think of it as a diagnostic report.

In [2]:
# Check for missing values (count + percentage per column)
missing_values = df.isnull().sum()#count the number of missing values in each column
missing_percentage = (df.isnull().sum() / len(df)) * 100 #calculate the percentage of missing values in each column

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_values.values,
    'Missing_Percentage': missing_percentage.values
}).sort_values('Missing_Count', ascending=False) #creates a DataFrame to summarize missing values and sort by count

print("Missing Values Report:") #print the missing values report showing count and percentage of missing values for each column
print(missing_df) #print the missing values DataFrame
print(f"\nTotal rows with at least one missing value: {df.isnull().any(axis=1).sum()}")#print the total number of rows that have at least one missing value across all columns(axis=1) checks for any missing value in the row and .sum() sums them up

Missing Values Report:
                                Column  Missing_Count  Missing_Percentage
0                              Country              0                 0.0
1                                 Year              0                 0.0
20                     Education Index              0                 0.0
19             Per Capita Income (USD)              0                 0.0
18          Improvement in 5 Years (%)              0                 0.0
17                               DALYs              0                 0.0
16                   Recovery Rate (%)              0                 0.0
15  Availability of Vaccines/Treatment              0                 0.0
14        Average Treatment Cost (USD)              0                 0.0
13                      Treatment Type              0                 0.0
12              Hospital Beds per 1000              0                 0.0
11                    Doctors per 1000              0                 0.0
10             

In [3]:
# Check for duplicate rows
duplicates = df.duplicated().sum() #counts the number of duplicate rows in the dataset using the duplicated() method which returns a boolean Series indicating duplicate rows and sum() counts them
print(f"Total duplicate rows: {duplicates}") #prints the total number of duplicate rows found in the dataset
print(f"Unique rows: {len(df) - duplicates}") #prints the number of unique rows by subtracting the number of duplicates from the total number of rows in the dataset
print(f"Duplicate ratio: {(duplicates / len(df) * 100):.2f}%") #prints the percentage of duplicate rows relative to the total number of rows in the dataset, formatted to 2 decimal places

# Show duplicate rows if any exist
if duplicates > 0:
    print("\nSample duplicate rows:")
    print(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10)) #if there are duplicate rows, this code will print a sample of them sorted by all columns; if not, it will confirm that no duplicates were found
else:
    print("\nNo duplicate rows found.")#if there are duplicate rows, this code will print a sample of them sorted by all columns; if not, it will confirm that no duplicates were found

Total duplicate rows: 0
Unique rows: 1000000
Duplicate ratio: 0.00%

No duplicate rows found.


In [4]:
# Check data types and flag any mismatches
print("Current Data Types:") #print the current data types of each column in the DataFrame to identify any potential issues with data types that may need to be addressed during cleaning and transformation
print(df.dtypes)#print the data types


# Identify potential type mismatches
dtype_issues = [] #initialize an empty list to store any identified data type issues for further review and cleaning

for col in df.columns:
    # Check if numeric columns are stored as object
    if 'rate' in col.lower() or 'percentage' in col.lower() or 'population' in col.lower(): #if the column name contains 'rate', 'percentage', or 'population', it is likely intended to be numeric, so we check if its data type is object (string) which would indicate a mismatch
        if df[col].dtype == 'object': #if the data type of the column is object, it suggests that numeric values may be stored as strings, which is a common issue when loading data from CSV files; we flag this as a potential data type issue for review and cleaning
            dtype_issues.append(f"'{col}' — should be numeric but is {df[col].dtype}") #if the column is expected to be numeric but is currently of type object, we add a message to the dtype_issues list indicating the column name and its current data type for further investigation and cleaning
    
    # Check if year columns are numeric
    if 'year' in col.lower():
        if df[col].dtype != 'int64' and df[col].dtype != 'float64':
            dtype_issues.append(f"'{col}' — should be numeric but is {df[col].dtype}")
    
    # Check for date-like columns stored as strings
    if 'date' in col.lower():
        if df[col].dtype == 'object':
            dtype_issues.append(f"'{col}' — should be datetime but is {df[col].dtype}")#if the column name contains 'date' and its data type is object, it suggests that date values may be stored as strings, which can cause issues for date-based analysis; we flag this as a potential data type issue for review and cleaning

print("\nData Type Issues Found:")
if dtype_issues:
    for issue in dtype_issues:
        print(f"  ⚠ {issue}")#if any potential data type issues were identified, we print them out with a warning symbol for visibility; if no issues were found, we confirm that no obvious type mismatches were detected         
else:
    print("  ✓ No obvious type mismatches detected.")



Current Data Types:
Country                                object
Year                                    int64
Disease Name                           object
Disease Category                       object
Prevalence Rate (%)                   float64
Incidence Rate (%)                    float64
Mortality Rate (%)                    float64
Age Group                              object
Gender                                 object
Population Affected                     int64
Healthcare Access (%)                 float64
Doctors per 1000                      float64
Hospital Beds per 1000                float64
Treatment Type                         object
Average Treatment Cost (USD)            int64
Availability of Vaccines/Treatment     object
Recovery Rate (%)                     float64
DALYs                                   int64
Improvement in 5 Years (%)            float64
Per Capita Income (USD)                 int64
Education Index                       float64
Urbanization R

In [5]:
# Detect outliers/anomalies in numerical columns (IQR or z-score method)
from scipy import stats #import stats module from scipy for z-score calculation for outlier detection

# Get numerical columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns #selects columns with numeric data types for outlier detection

print("OUTLIER DETECTION — IQR Method")#print a header for the IQR method of outlier detection
print("="*60) # underlines header with equal signs for emphasis

outlier_summary = [] #initialize an empty list to store summary of outliers detected in each numeric column for reporting

for col in numeric_cols:
    Q1 = df[col].quantile(0.25) #calculate the first quartile (25th percentile) for the column to determine the lower bound for outlier detection to identify values that are significantly lower than the majority of the data
    Q3 = df[col].quantile(0.75)#calculate the third quartile (75th percentile) for the column to determine the upper bound for outlier detection to identify values that are significantly higher than the majority of the data
    IQR = Q3 - Q1 #calculate the interquartile range (IQR) which is the difference between the third and first quartiles; it represents the range of the middle 50% of the data and is used to identify outliers that fall outside of this range
    
    lower_bound = Q1 - 1.5 * IQR#calculate the lower bound for outlier detection using the IQR method; any value below this threshold is considered a potential outlier
    upper_bound = Q3 + 1.5 * IQR#calculate the upper bound for outlier detection using the IQR method; any value above this threshold is considered a potential outlier
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)] #identify outliers in the column by filtering the DataFrame for values that are either below the lower bound or above the upper bound
    outlier_count = len(outliers)#count the number of outliers detected in the column by calculating the length of the filtered DataFrame containing only the outliers
    outlier_pct = (outlier_count / len(df)) * 100 #calculate the percentage of outliers relative to the total number of rows in the dataset for the column being analyzed
    
    if outlier_count > 0:
        outlier_summary.append({
            'Column': col,
            'Outlier_Count': outlier_count,
            'Outlier_Percentage': outlier_pct,
            'Lower_Bound': lower_bound,
            'Upper_Bound': upper_bound
        })#if any outliers were detected in the column, we append a summary of the outlier count, percentage, and bounds to the outlier_summary list for reporting; if no outliers were found, we simply move on to the next column

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary).sort_values('Outlier_Count', ascending=False)
    print(outlier_df.to_string(index=False))
else:
    print("✓ No outliers detected using IQR method.")#if no outliers were detected using the IQR method, we print a confirmation message; if outliers were found, we create a DataFrame from the summary and print it sorted by the count of outliers in descending order for easy identification of columns with the most outliers

# Z-score method (values with |z-score| > 3 are extreme outliers)
print("\n" + "="*60)#prints 60 equal signs to act as a line to separate sections with equal signs for emphasis
print("OUTLIER DETECTION — Z-Score Method (|z| > 3)")#  prints a header for the z-score method of outlier detection, indicating that we will be identifying extreme outliers based on their z-scores being greater than 3 in absolute value, which means they are more than 3 standard deviations away from the mean
print("="*60)#prints 60 equal signs to act as a line to separate sections with equal signs for emphasis

z_outliers = []
for col in numeric_cols:
    z_scores = np.abs(stats.zscore(df[col].dropna()))#calculate the z-scores for the column by standardizing the values (subtracting the mean and dividing by the standard deviation) and taking the absolute value to identify how many standard deviations away each value is from the mean; we drop NA values before calculating z-scores to avoid errors from missing data
    extreme_outliers = (z_scores > 3).sum()#calculate the number of extreme outliers in the column by counting how many z-scores have an absolute value greater than 3, which indicates that those values are significantly different from the mean and may be considered anomalies(statistics: in a roughly normal distribution, about 99.7% of values fall within ±3 standard deviations.); we drop NA values before calculating z-scores to avoid errors from missing data
    if extreme_outliers > 0:
        z_outliers.append({
            'Column': col,
            'Extreme_Outliers_(z>3)': extreme_outliers,
            'Percentage': (extreme_outliers / len(df)) * 100
        })#if any extreme outliers were detected in the column based on the z-score method, we append a summary of the count and percentage of extreme outliers to the z_outliers list for reporting; if no extreme outliers were found, we simply move on to the next column

if z_outliers:
    z_df = pd.DataFrame(z_outliers).sort_values('Extreme_Outliers_(z>3)', ascending=False)#if extreme outliers were found using the z-score method, we create a DataFrame from the summary and print it sorted by the count of extreme outliers in descending order for easy identification of columns with the most extreme outliers
    print(z_df.to_string(index=False))
else:
    print("✓ No extreme outliers detected using z-score method.")#if no extreme outliers were detected using the z-score method, we print a confirmation message; if extreme outliers were found, we create a DataFrame from the summary and print it sorted by the count of extreme outliers in descending order for easy identification of columns with the most extreme outliers

OUTLIER DETECTION — IQR Method
✓ No outliers detected using IQR method.

OUTLIER DETECTION — Z-Score Method (|z| > 3)
✓ No extreme outliers detected using z-score method.


In [6]:
# Check for invalid values (e.g., rates outside 0-100, negative populations)
print("INVALID VALUES CHECK")#print a header for the invalid values check section to indicate that we will be looking for values that are outside of expected ranges, such as rates that should be between 0 and 100 or population counts that should not be negative
print("="*60)#prints 60 equal signs to act as a line to separate sections with equal signs for emphasis

invalid_issues = []#initialize an empty list to store any identified invalid value issues for further review and cleaning

for col in df.columns:
    # Check for negative values in columns that should be non-negative (only numeric columns)
    if ('population' in col.lower() or 'count' in col.lower() or 'cases' in col.lower()) and df[col].dtype in ['float64', 'int64']:
        negative_count = (df[col] < 0).sum() #calculate the number of negative values in the column by filtering for values less than 0 and counting them; this is relevant for columns that represent population counts, case counts, or any other metric that should logically be non-negative
        if negative_count > 0:
            invalid_issues.append({
                'Column': col,
                'Issue': 'Negative values (should be ≥0)',
                'Count': negative_count,
                'Percentage': (negative_count / len(df)) * 100
            }) #if any negative values were detected in the column, we append a summary of the count and percentage of negative values to the invalid_issues list for reporting; if no negative values were found, we simply move on to the next column
    
    # Check for percentages/rates outside 0-100 range
    if ('rate' in col.lower() or 'percentage' in col.lower() or '(%)' in col) and df[col].dtype in ['float64', 'int64']: #if the column name suggests it should be a rate or percentage and its data type is numeric, we check for values that are outside the expected range of 0 to 100, which would indicate invalid data for rates and percentages
        below_0 = (df[col] < 0).sum() #calculate the number of values in the column that are below 0
        above_100 = (df[col] > 100).sum()#calculate the number of values in the column that are above 100
        
        if below_0 > 0:
            invalid_issues.append({
                'Column': col,
                'Issue': 'Below 0 (should be 0-100)',
                'Count': below_0,
                'Percentage': (below_0 / len(df)) * 100
            })#if any values below 0 were detected in the column, we append a summary of the count and percentage of values below 0 to the invalid_issues list for reporting; if no values below 0 were found, we simply move on to check for values above 100
        if above_100 > 0:
            invalid_issues.append({
                'Column': col,
                'Issue': 'Above 100 (should be 0-100)',
                'Count': above_100,
                'Percentage': (above_100 / len(df)) * 100
            })#if any values above 100 were detected in the column, we append a summary of the count and percentage of values above 100 to the invalid_issues list for reporting; if no values above 100 were found, we simply move on to the next column

if invalid_issues:
    invalid_df = pd.DataFrame(invalid_issues).sort_values('Count', ascending=False)
    print(invalid_df.to_string(index=False))
else:
    print("✓ No invalid values detected.")#if no invalid values were detected based on the checks for negative values and rates outside the 0-100 range, we print a confirmation message; if any issues were found, we create a DataFrame from the summary and print it sorted by the count of issues in descending order for easy identification of columns with the most invalid values

# Show sample statistics for key numeric columns
print("\n" + "="*60)
print("Numeric Column Statistics (Quick Check):")#print a header for the numeric column statistics section to indicate that we will be providing a quick summary of key statistics (min and max) for numeric columns to help identify any potential issues with the range of values in those columns
print("="*60)
numeric_stats = df.select_dtypes(include=[np.number]).describe().loc[['min', 'max']].T#selects numeric columns, calculates descriptive statistics, and extracts the minimum and maximum values for each numeric column to provide a quick check of the range of values in those columns, which can help identify any potential issues such as unexpected ranges or outliers, .T transposes the result so each row is one column from df, with min and max as columns.
# verifies whether numeric columns have expected ranges.
print(numeric_stats)

INVALID VALUES CHECK
✓ No invalid values detected.

Numeric Column Statistics (Quick Check):
                                 min        max
Year                          2000.0     2024.0
Prevalence Rate (%)              0.1       20.0
Incidence Rate (%)               0.1       15.0
Mortality Rate (%)               0.1       10.0
Population Affected           1000.0  1000000.0
Healthcare Access (%)           50.0      100.0
Doctors per 1000                 0.5        5.0
Hospital Beds per 1000           0.5       10.0
Average Treatment Cost (USD)   100.0    50000.0
Recovery Rate (%)               50.0       99.0
DALYs                            1.0     5000.0
Improvement in 5 Years (%)       0.0       10.0
Per Capita Income (USD)        500.0   100000.0
Education Index                  0.4        0.9
Urbanization Rate (%)           20.0       90.0


**Audit Summary** *(fill in after running the cells above)*

| Issue | Column(s) Affected | Count | Action Planned |
|---|---|---|---|
| Missing values | | | |
| Duplicate rows | — | | |
| Outliers | | | |
| Invalid values | | | |
| Type mismatches | | | |

---
## 3. Data Cleaning & Standardization

**What to do here:**
- Handle **missing values** (impute, drop, or flag — justify your choice)
- Remove or correct **duplicate rows**
- Handle **outliers** (cap, remove, or transform — justify your choice)
- Fix **data type mismatches** (e.g., convert columns to correct types)
- Standardize **string/categorical values** (e.g., consistent casing, strip whitespace)
- Correct any **invalid values** found in the audit

**Deliverable check:** Each cleaning step should be a reusable function where possible, making the pipeline reproducible.

In [7]:
# Handle missing values
# Strategy:
#   - Numeric columns: impute with median (robust to outliers)
#   - Categorical columns: impute with mode (most frequent value)
#   - Create indicator columns for rows that had missing values (for feature engineering)

df_cleaned = df.copy()#create a copy of the original DataFrame to perform cleaning and imputation without modifying the original data, allowing us to keep the raw data intact for reference or if we need to revert back to it later

# Track which rows had missing values
df_cleaned['Had_Missing_Values'] = df.isnull().any(axis=1).astype(int)#create a new column 'Had_Missing_Values' that flags rows with any missing values across all columns; this can be useful for feature engineering later on, as it allows us to identify which rows had missing data before we perform imputation

print("Missing Values Handling:")
print("="*60)#print a header for the missing values handling section to indicate that we will be addressing missing values in the dataset using imputation strategies based on the data type of the columns, and we will also be creating an indicator column to flag rows that had missing values for potential use in feature engineering

# Identify columns with missing values
cols_with_missing = df_cleaned.isnull().sum()#count the number of missing values in each column to identify which columns have missing data that needs to be addressed through imputation; this will help us focus our cleaning efforts on the relevant columns
cols_with_missing = cols_with_missing[cols_with_missing > 0]#filter the Series to include only columns that have more than 0 missing values, so we can focus our imputation efforts on those specific columns; if no columns have missing values, this will result in an empty Series

if len(cols_with_missing) > 0:
    print(f"Columns with missing values: {list(cols_with_missing.index)}") #if there are columns with missing values, we print a list of those column names to indicate which columns we will be addressing with imputation; if no columns have missing values, we will confirm that no imputation is needed
    
    for col in cols_with_missing.index:
        if df_cleaned[col].dtype in ['float64', 'int64']:#if the data type of the column is numeric, we will impute missing values using the median, which is a robust measure of central tendency that is less affected by outliers compared to the mean; if the column is categorical, we will impute using the mode, which is the most frequently occurring value in the column
            # Numeric: impute with median
            median_val = df_cleaned[col].median()#calculate the median value for the column to use for imputation of missing values in numeric columns, as the median is less sensitive to outliers compared to the mean and provides a better central value for skewed distributions
            df_cleaned[col].fillna(median_val, inplace=True)#impute missing values in the numeric column with the calculated median value using the fillna() method, which replaces NA values with the specified value; inplace=True modifies the DataFrame in place without needing to assign it back to df_cleaned
            print(f"  • {col}: Imputed {cols_with_missing[col]} values with median ({median_val:.2f})")#if the column is numeric, we impute missing values with the median and print a message indicating how many values were imputed and what the median value used for imputation was, formatted to 2 decimal places for readability
        else:
            # Categorical: impute with mode
            mode_val = df_cleaned[col].mode()[0]#calculate the mode value for the column to use for imputation of missing values in categorical columns, as the mode represents the most frequently occurring value and is a common choice for imputation of categorical data
            df_cleaned[col].fillna(mode_val, inplace=True)#impute missing values in the categorical column with the calculated mode value using the fillna() method, which replaces NA values with the specified value; inplace=True modifies the DataFrame in place without needing to assign it back to df_cleaned
            print(f"  • {col}: Imputed {cols_with_missing[col]} values with mode ('{mode_val}')")# if the column is categorical, we impute missing values with the mode and print a message indicating how many values were imputed and what the mode value used for imputation was
else:
    print("✓ No missing values found — no imputation needed.")

print("\n" + "="*60)
print(f"Rows with missing values flag: {df_cleaned['Had_Missing_Values'].sum()}")#print the total number of rows that were flagged as having had missing values before imputation, which can be useful for understanding how many rows were affected by missing data and may be relevant for feature engineering or analysis later on
print(f"Remaining missing values in dataset: {df_cleaned.isnull().sum().sum()}")#print the total number of remaining missing values in the cleaned dataset after imputation to confirm that all missing values have been addressed; this should ideally be 0 if all missing values were successfully imputed, and it serves as a final check on our missing value handling process


Missing Values Handling:
✓ No missing values found — no imputation needed.

Rows with missing values flag: 0
Remaining missing values in dataset: 0


In [ ]:
# Remove duplicate rows
print("Duplicate Rows Removal:")#print a header for the duplicate rows removal section to indicate that we will be addressing duplicate rows in the dataset by removing exact duplicates where all columns match, and we will report on the number of duplicates removed and the impact on the dataset size
print("="*60)

rows_before = len(df_cleaned)#store the number of rows in the cleaned DataFrame before removing duplicates so we can compare it to the number of rows after deduplication to understand how many duplicate rows were removed and the impact on the dataset size

# Remove exact duplicates (all columns match)
df_cleaned = df_cleaned.drop_duplicates()#remove duplicate rows from the cleaned DataFrame using the drop_duplicates() method, which by default considers all columns to identify duplicates; this will keep the first occurrence of each unique row and remove subsequent duplicates, resulting in a DataFrame with only unique rows based on all column values

rows_after = len(df_cleaned) #store the number of rows in the cleaned DataFrame after removing duplicates so we can compare it to the number of rows before deduplication to understand how many duplicate rows were removed and the impact on the dataset size
rows_removed = rows_before - rows_after #calculate the number of duplicate rows removed by subtracting the number of rows after deduplication from the number of rows before deduplication; this gives us a count of how many duplicate rows were identified and removed from the dataset, which is important for understanding the quality and uniqueness of the data we are working with

print(f"Rows before deduplication: {rows_before:,}")#print the number of rows in the dataset before deduplication, formatted with commas for readability, to provide context on the size of the dataset before we removed duplicate rows
print(f"Rows after deduplication: {rows_after:,}")#print the number of rows in the dataset after deduplication, formatted with commas for readability, to provide context on the size of the dataset after we removed duplicate rows and to show the impact of the deduplication process
print(f"Duplicate rows removed: {rows_removed:,}")#print the number of duplicate rows that were removed, formatted with commas for readability, to highlight the number of duplicates that were identified and removed from the dataset, which is important for understanding the quality and uniqueness of the data we are working with
print(f"Duplicate ratio: {(rows_removed / rows_before * 100):.2f}%")#print the percentage of duplicate rows that were removed relative to the total number of rows before deduplication, formatted to 2 decimal places, to provide insight into how prevalent duplicate rows were in the original dataset and the impact of the deduplication process on the overall data quality

if rows_removed > 0:
    print(f"\n✓ Successfully removed {rows_removed:,} duplicate rows.")
else:
    print("\n✓ No duplicate rows found.") #if any duplicate rows were removed, we print a confirmation message indicating how many duplicate rows were successfully removed; if no duplicate rows were found, we print a message confirming that no duplicates were detected in the dataset     

Duplicate Rows Removal:
Rows before deduplication: 1,000,000
Rows after deduplication: 1,000,000
Duplicate rows removed: 0
Duplicate ratio: 0.00%

✓ No duplicate rows found.


In [ ]:
# Handle outliers
# Approach: Cap at IQR bounds (preserves data volume while reducing extreme values)
# Justification: Capping is preferred over removal to maintain sample size; more robust than winsorization for extreme outliers

print("Outlier Handling:")#print a header for the outlier handling section to indicate that we will be addressing outliers in the dataset by capping values at the IQR bounds, which allows us to reduce the influence of extreme values while preserving the overall sample size and distribution of the data, and we will report on the columns affected and the number of outliers capped
print("="*60)

# Track rows with outliers
df_cleaned['Had_Outliers'] = False #create a new column 'Had_Outliers' that will be used to flag rows that had outliers before we cap them, which can be useful for feature engineering later on, as it allows us to identify which rows had extreme values that were adjusted during the outlier handling process

numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns #selects columns with numeric data types for outlier handling, as outliers are typically identified and handled in numeric columns; this will allow us to focus our outlier capping efforts on the relevant columns that contain numeric data where outliers may be present
outlier_caps_applied = []#initialize an empty list to store a summary of the outlier capping applied to each numeric column, including the number of outliers capped and the bounds used for capping, which will be useful for reporting and understanding the impact of the outlier handling process on the dataset

for col in numeric_cols: #iterate over each numeric column in the cleaned DataFrame to identify and cap outliers based on the IQR method; we will calculate the IQR bounds for each column, identify outliers, flag rows with outliers, and cap values at the bounds while keeping track of the number of outliers capped for reporting purposes
    # Skip the indicator columns we added
    if col in ['Had_Missing_Values', 'Had_Outliers']:#if the column is one of the indicator columns we added for tracking missing values or outliers, we skip it in the outlier handling process since these columns are not meant to be analyzed for outliers and should not be modified; this ensures that we only apply outlier capping to the relevant numeric columns in the dataset
        continue
    
    Q1 = df_cleaned[col].quantile(0.25)#calculate the first quartile (25th percentile) for the column to determine the lower bound for outlier detection and capping; this helps us identify values that are significantly lower than the majority of the data in the column
    Q3 = df_cleaned[col].quantile(0.75)#calculate the third quartile (75th percentile) for the column to determine the upper bound for outlier detection and capping; this helps us identify values that are significantly higher than the majority of the data in the column
    IQR = Q3 - Q1#calculate the interquartile range (IQR) for the column, which is the difference between the third and first quartiles; the IQR represents the range of the middle 50% of the data and is used to identify outliers that fall outside of this range
    
    lower_bound = Q1 - 1.5 * IQR #calculate the lower bound for outlier detection and capping using the IQR method; any value below this threshold is considered a potential outlier and will be capped at this lower bound to reduce the influence of extreme low values while preserving the overall distribution of the data
    upper_bound = Q3 + 1.5 * IQR #calculate the upper bound for outlier detection and capping using the IQR method; any value above this threshold is considered a potential outlier and will be capped at this upper bound to reduce the influence of extreme high values while preserving the overall distribution of the data
    
    # Identify outliers before capping
    outliers_below = (df_cleaned[col] < lower_bound).sum()#calculate the number of outliers below the lower bound by filtering the column for values less than the lower bound and counting them; this gives us a count of how many values in the column are considered outliers on the low end before we apply capping
    outliers_above = (df_cleaned[col] > upper_bound).sum()#calculate the number of outliers above the upper bound by filtering the column for values greater than the upper bound and counting them; this gives us a count of how many values in the column are considered outliers on the high end before we apply capping
    
    if outliers_below > 0 or outliers_above > 0:
        # Mark rows with outliers
        df_cleaned.loc[(df_cleaned[col] < lower_bound) | (df_cleaned[col] > upper_bound), 'Had_Outliers'] = True #flag rows in the cleaned DataFrame that have outliers in the current column by setting the 'Had_Outliers' column to True for those rows; this allows us to keep track of which rows had extreme values before we cap them, which can be useful for feature engineering or analysis later on
        
        # Cap values at bounds
        df_cleaned[col] = df_cleaned[col].clip(lower_bound, upper_bound) #cap values in the column at the calculated lower and upper bounds using the clip() method, which replaces values below the lower bound with the lower bound and values above the upper bound with the upper bound; this reduces the influence of extreme outliers while preserving the overall distribution of the data
        
        outlier_caps_applied.append({
            'Column': col,
            'Below_Lower_Bound': outliers_below,
            'Above_Upper_Bound': outliers_above,
            'Total_Capped': outliers_below + outliers_above,
            'Lower_Bound': lower_bound,
            'Upper_Bound': upper_bound
        })#if any outliers were detected in the column, we append a summary of the outlier capping applied to that column to the outlier_caps_applied list, including the count of outliers below the lower bound, above the upper bound, the total number of outliers capped, and the bounds used for capping; this will be useful for reporting and understanding the impact of the outlier handling process on each column in the dataset

if outlier_caps_applied:
    outlier_caps_df = pd.DataFrame(outlier_caps_applied)
    print(outlier_caps_df.to_string(index=False))
    print(f"\n✓ Successfully capped {outlier_caps_df['Total_Capped'].sum():,} outlier values across {len(outlier_caps_df)} columns.")
else:
    print("✓ No outliers detected — no capping needed.")#if any outliers were capped, we create a DataFrame from the summary of outlier capping applied and print it for review, along with a confirmation message indicating how many outlier values were capped across how many columns; if no outliers were detected, we print a message confirming that no capping was needed

print("\n" + "="*60)
print(f"Rows with outliers capped: {df_cleaned['Had_Outliers'].sum():,}")#print the total number of rows that were flagged as having had outliers before capping, which can be useful for understanding how many rows were affected by outliers and may be relevant for feature engineering or analysis later on; this gives us insight into the prevalence of outliers in the dataset and the impact of our outlier handling process

Outlier Handling:
✓ No outliers detected — no capping needed.

Rows with outliers capped: 0


In [ ]:
# Fix data types (e.g., Year to int, categorical columns to category dtype)
print("Data Type Conversion:")#print a header for the data type conversion section to indicate that we will be addressing data type issues in the dataset by converting columns to more appropriate data types, such as converting year columns to integers and categorical columns to the category dtype, which can help reduce memory usage and improve performance for analysis and modeling
print("="*60)

dtypes_before = df_cleaned.dtypes.copy()#store a copy of the data types before conversion so we can compare it to the data types after conversion to understand which columns were converted and how the data types changed as a result of our cleaning and transformation efforts

# Convert Year to int
if 'Year' in df_cleaned.columns:#if there is a column named 'Year' in the cleaned DataFrame, we will attempt to convert it to an integer data type, as year values are typically represented as integers and this can help reduce memory usage and improve performance for analysis and modeling; if the 'Year' column is not present, we will skip this conversion step
    df_cleaned['Year'] = df_cleaned['Year'].astype('int64')#convert the 'Year' column to the int64 data type using the astype() method, which will change the data type of the column to integer, allowing for more efficient storage and faster computations when working with year values in analysis and modeling
    print("✓ Year converted to int64")#if the 'Year' column was successfully converted to int64, we print a confirmation message indicating that the conversion was successful; if the 'Year' column was not present, we will have skipped this step and not printed this message

# Identify and convert categorical columns (object dtype columns with high cardinality are typically categorical)
categorical_cols = df_cleaned.select_dtypes(include='object').columns.tolist()

# Remove indicator columns from categorical conversion
categorical_cols = [col for col in categorical_cols if col not in ['Had_Missing_Values', 'Had_Outliers']]

conversions = []#initialize an empty list to store a summary of the categorical columns that were converted to the category data type, including the original data type, new data type, and the number of unique values, which will be useful for reporting and understanding the impact of our data type conversion efforts on memory usage and performance
for col in categorical_cols:#iterate over each column that has an object data type to identify potential categorical columns based on their unique value counts and convert them to the category data type if they meet the criteria for being considered categorical, which can help reduce memory usage and improve performance for analysis and modeling when working with categorical data
    unique_count = df_cleaned[col].nunique()#calculate the number of unique values in the column using the nunique() method, which gives us insight into the cardinality of the column and helps us determine whether it is suitable for conversion to the category data type; columns with a high number of unique values may not be ideal candidates for category conversion due to potential memory issues, while columns with a low number of unique values are often good candidates for category conversion
    total_rows = len(df_cleaned)#store the total number of rows in the cleaned DataFrame to calculate the percentage of unique values in the column relative to the total number of rows, which can help us determine whether the column is suitable for conversion to the category data type based on a typical threshold for categorical data (e.g., if unique values are less than 20% of total rows, it is often considered a good candidate for category conversion)
    uniqueness_pct = (unique_count / total_rows) * 100 #calculate the percentage of unique values in the column relative to the total number of rows by dividing the unique count by the total number of rows and multiplying by 100; this gives us a measure of how many unique categories are present in the column compared to the overall size of the dataset, which can help us determine whether it is suitable for conversion to the category data type based on a typical threshold for categorical data (e.g., if unique values are less than 20% of total rows, it is often considered a good candidate for category conversion)
    
    # Convert to category if unique values < 20% of total rows (typical threshold for categorical data)
    if uniqueness_pct < 20:#if the percentage of unique values in the column is less than 20%, we consider it a good candidate for conversion to the category data type, as it likely represents categorical data with a manageable number of unique categories; if the uniqueness percentage is 20% or higher, we will skip the conversion for that column to avoid potential memory issues associated with high cardinality categorical data
        df_cleaned[col] = df_cleaned[col].astype('category')#convert the column to the category data type using the astype() method, which will change the data type of the column to category, allowing for more efficient storage and faster computations when working with categorical data in analysis and modeling
        conversions.append({
            'Column': col,
            'Original_Type': 'object',
            'New_Type': 'category',
            'Unique_Values': unique_count
        })#if the column was successfully converted to the category data type, we append a summary of the conversion to the conversions list, including the column name, original data type, new data type, and the number of unique values in the column; this will be useful for reporting and understanding which columns were converted and the impact of those conversions on memory usage and performance

if conversions:
    conversions_df = pd.DataFrame(conversions)#if any columns were converted to the category data type, we create a DataFrame from the summary of conversions and print it for review, along with a confirmation message indicating how many columns were converted; if no columns were eligible for conversion based on the uniqueness percentage threshold, we will print a message confirming that no conversions were made due to high cardinality
    print("\nCategorical Columns Converted:")
    print(conversions_df.to_string(index=False))#print the DataFrame containing the summary of categorical columns that were converted to the category data type, including the column name, original data type, new data type, and the number of unique values in each converted column, for review and understanding of the impact of our data type conversion efforts on memory usage and performance
else:
    print("\nNo columns eligible for category conversion (high cardinality).")#if no columns were eligible for conversion to the category data type due to having a high percentage of unique values, we print a message confirming that no conversions were made due to high cardinality, which indicates that all object dtype columns had too many unique values to be efficiently converted to the category data type without potentially increasing memory usage

# Show memory usage before and after
print("\n" + "="*60)#print a header for the memory usage comparison section to indicate that we will be comparing the memory usage of the dataset before and after our cleaning and transformation efforts, including data type conversions, to understand the impact of our optimizations on memory efficiency
print("Memory Usage Comparison:")#
dtypes_after = df_cleaned.dtypes#store the data types after conversion so we can compare it to the data types before conversion to understand which columns were converted and how the data types changed as a result of our cleaning and transformation efforts; this will help us understand the impact of our data type conversions on memory usage and performance
memory_before = df.memory_usage(deep=True).sum() / 1024**2#calculate the total memory usage of the original DataFrame before cleaning and transformation by summing the memory usage of all columns using the memory_usage() method with deep=True to get an accurate measurement of memory usage for object dtype columns, and then converting it to megabytes by dividing by 1024 squared; this gives us a baseline for understanding how much memory the original dataset was using before we applied our optimizations
memory_after = df_cleaned.memory_usage(deep=True).sum() / 1024**2#calculate the total memory usage of the cleaned DataFrame after cleaning and transformation by summing the memory usage of all columns using the memory_usage() method with deep=True to get an accurate measurement of memory usage for object dtype columns, and then converting it to megabytes by dividing by 1024 squared; this allows us to compare the memory usage before and after our optimizations to understand the impact of our cleaning and data type conversions on memory efficiency
memory_saved = memory_before - memory_after#calculate the amount of memory saved by subtracting the memory usage after cleaning and transformation from the memory usage before cleaning and transformation; this gives us a measure of how much memory we were able to reduce through our optimizations, which is important for understanding the efficiency of our data handling and the potential benefits for analysis and modeling performance
memory_saved_pct = (memory_saved / memory_before) * 100 #calculate the percentage of memory saved by dividing the memory saved by the memory usage before cleaning and transformation and multiplying by 100; this gives us a relative measure of how much memory we were able to reduce compared to the original dataset, which can be useful for understanding the efficiency of our optimizations in terms of memory usage

print(f"Before optimization: {memory_before:.2f} MB") #print the memory usage of the original dataset before cleaning and transformation, formatted to 2 decimal places for readability, to provide context on how much memory the original dataset was using before we applied our optimizations
print(f"After optimization:  {memory_after:.2f} MB")  #print the memory usage of the cleaned dataset after cleaning and transformation, formatted to 2 decimal places for readability, to provide context on how much memory the cleaned dataset is using after we applied our optimizations and to allow for comparison with the original memory usage
print(f"Memory saved:        {memory_saved:.2f} MB ({memory_saved_pct:.1f}%)") #print the amount of memory saved through our optimizations, formatted to 2 decimal places for readability, along with the percentage of memory saved relative to the original dataset, formatted to 1 decimal place, to highlight the impact of our cleaning and data type conversions on memory efficiency and to provide insight into the benefits of our optimizations for analysis and modeling performance

print("\n" + "="*60) #print a header for the final data types section to indicate that we will be showing the final data types of the columns in the cleaned dataset after all of our cleaning and transformation efforts, which can help us understand the structure of the cleaned data and confirm that our data type conversions were applied as expected
print("Final Data Types:") #print the final data types of the columns in the cleaned DataFrame to provide insight into the structure of the cleaned dataset and confirm that our data type conversions were applied as expected, which can help us understand how the data is now organized and how it may be more efficient for analysis and modeling
print(df_cleaned.dtypes) #print the data types of each column in the cleaned DataFrame to show the final structure of the cleaned dataset, including any changes that were made to data types during our cleaning and transformation process, which can help us understand how the data is now organized and how it may be more efficient for analysis and modeling

Data Type Conversion:
✓ Year converted to int64

No columns eligible for category conversion (high cardinality).

Memory Usage Comparison:
Before optimization: 484.66 MB
After optimization:  129.71 MB
Memory saved:        354.96 MB (73.2%)

Final Data Types:
Country                               category
Year                                     int64
Disease Name                          category
Disease Category                      category
Prevalence Rate (%)                    float64
Incidence Rate (%)                     float64
Mortality Rate (%)                     float64
Age Group                             category
Gender                                category
Population Affected                      int64
Healthcare Access (%)                  float64
Doctors per 1000                       float64
Hospital Beds per 1000                 float64
Treatment Type                        category
Average Treatment Cost (USD)             int64
Availability of Vaccines/Treatment  

In [ ]:
# Standardize string/categorical values (strip whitespace, consistent casing)


In [ ]:
# Verify cleaning results — rerun the audit checks and confirm issues are resolved


---
## 4. Transformation: Joins, Grouping & Aggregations

**What to do here:**
- Create **grouped summaries** (e.g., average mortality rate by country, by disease category, by year) => Like those in the CATs
- Perform **aggregations** that will be useful for visualization (e.g., total population affected per region/year)
- If working with multiple data sources, perform **joins/merges** here
- Create any **pivot tables** or **cross-tabulations** that reveal patterns

**Deliverable check:** Covers the "Transformation (aggregation, grouping, joins)" requirement.

In [ ]:
# Group by Country — compute mean Mortality Rate, Recovery Rate, Healthcare Access


In [ ]:
# Group by Disease Category — compute summary statistics


In [ ]:
# Group by Year — compute trends over time


In [ ]:
# Any joins/merges with additional data sources (if applicable)


In [ ]:
# Pivot table or cross-tabulation (e.g., Disease Category x Age Group)


---
## 5. Feature Engineering

**What to do here:**
- Create **new derived columns** that add analytical value, for example:
  - `Mortality_to_Prevalence_Ratio` = Mortality Rate / Prevalence Rate
  - `Healthcare_Gap` = 100 - Healthcare Access (%)
  - `High_Mortality_Flag` = 1 if Mortality Rate > threshold, else 0
  - `Decade` = Year grouped into decades (2000s, 2010s, 2020s)
  - Encoded versions of categorical columns (label encoding or one-hot)
- Document **why** each feature was created and what it represents

**Deliverable check:** Explicitly addresses the "Feature engineering" requirement.

In [ ]:
# Create ratio/derived numerical features
# Example: df['Mortality_to_Prevalence_Ratio'] = df['Mortality Rate (%)'] / df['Prevalence Rate (%)']


In [ ]:
# Create binary flag features (e.g., High_Mortality_Flag, Vaccine_Available_Flag)


In [ ]:
# Create time-based features (e.g., Decade, Years_Since_2000)


In [ ]:
# Encode categorical columns for future modeling (label encoding or one-hot)


In [ ]:
# Preview the final dataframe with all new features


---
## 6. Final Pipeline & Dataset Export

**What to do here:**
- Print a final summary of the cleaned + engineered dataset (shape, columns, dtypes)
- Export the processed dataset to `data/processed/` as a CSV file
- Write a **Data Dictionary** table below documenting every transformation made

**Deliverable check:** Delivers the "Clean and structured dataset" and "Documentation of transformations".

In [ ]:
# Print final dataset summary (shape, dtypes, head)


In [ ]:
# Export cleaned dataset to data/processed/
# Example:
# PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'global_health_cleaned.csv'
# df_clean.to_csv(PROCESSED_PATH, index=False)
# print(f"Saved to {PROCESSED_PATH}")


### Data Dictionary — Transformations Applied

*(Fill in after completing all sections above)*

| Column / Feature | Type | Description | Transformation Applied |
|---|---|---|---|
| *(original column)* | *(dtype)* | *(what it represents)* | *(e.g., no change / imputed mean / type cast)* |
| *(new feature)* | *(dtype)* | *(what it represents)* | *(e.g., derived from X / Y)* |

### Pipeline Summary

*(Fill in after completing all sections above)*

| Step | Action | Rows Before | Rows After | Columns Before | Columns After |
|---|---|---|---|---|---|
| Raw load | — | | | | |
| Remove duplicates | | | | | |
| Handle missing values | | | | | |
| Handle outliers | | | | | |
| Feature engineering | | | | | |
| **Final export** | | | | | |